# `ptof_obs_mal_output`

## What this notebook does
Detects three distinct ways an agent call can be malformed even when it "succeeds" in the
transport sense: **blank output** (nothing usable came back), **runtime/transport violations**
(the call used a transport/model_config/scheduler_run combination that isn't sanctioned for that
capability), and **response schema drift** (an expected output field silently stopped appearing).

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `03_malformed_output` -- runs in parallel with
  `02_latency_detection`, `04_hallucination_detection`, `05_behavioral_correlation`, after
  `01_bronze_projections`, before `06_alert`.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`),
  `runtime_allowlist`/`capability_registry` (human-curated by `ptof_obs_setup_seed`), and
  `response_schema_baseline` implicitly via the baseline-vs-current comparison in the last cell
  (built nightly by `ptof_obs_nightly_baseline` -- note this notebook actually recomputes the
  30-day baseline keys inline rather than reading the baseline table directly; see that cell's
  comments).
- **Downstream:** `ptof_obs_alert.ipynb` reads `blank_output_findings` (v1 detector
  `blank_output`, CRITICAL) and `response_schema_drift` (v1 detector `schema_field_missing`,
  CRITICAL, filtered to `drift_type = 'field_missing'`). `transport_violation_signatures` feeds
  the `runtime_violation` detector, currently silenced to WARN (registry drift means
  `runtime_allowlist`/`capability_registry` aren't trustworthy ground truth yet -- see the
  implementation plan's Phase 4).

## Tables/views touched
- **Reads:** `v_llm_bronze`, `runtime_allowlist`, `capability_registry`,
  `transport_violations` (this notebook's own output, read back for signature dedup).
- **Writes:** `blank_output_incidents` (per-hour blank-rate rollup), `blank_output_findings`
  (window-aggregated, signature-keyed -- what `ptof_obs_alert` reads), `transport_violations`
  (per-call flagged configuration violations), `transport_violation_signatures`
  (deduped-by-configuration, MERGE-safe version), `response_schema_drift` (baseline-vs-current
  field presence comparison, signature-keyed).


In [0]:
# env widget: which environment (dev/prod) this run targets -- read by the
# transport_violations cell below via the :env SQL parameter binding.
dbutils.widgets.text("env", "dev")

In [0]:
%sql
-- blank_output_incidents -- per-hour rollup of how often a capability returned a blank response
-- this run. Raw detail table; blank_output_findings (next cell) aggregates this into the actual
-- incident ptof_obs_alert reads.
-- Scoped to active capabilities via capability_registry join so deactivated DSA/probe
-- capabilities don't produce blank-output findings.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.blank_output_incidents AS
SELECT
    date_trunc('HOUR', b.called_at) AS hour,
    b.capability, b.model_config, b.transport,
    count_if(b.is_blank_output) AS blank_output_count,
    count(*)                    AS total_successful_calls,
    count_if(b.is_blank_output) * 1.0 / count(*) AS blank_output_rate
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.success = true
  AND b.is_credential_fastfail = false
GROUP BY 1, 2, 3, 4
HAVING count_if(b.is_blank_output) > 0;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- blank_output_findings — aggregated over 6h window for MERGE dedup. This is what
-- ptof_obs_alert.ipynb's blank_output detector (CRITICAL in v1) actually reads -- catches "the
-- call formally succeeded but returned nothing," a failure class invisible to every
-- error-rate/success-flag-based detector in the system.
-- Key on (capability, model_config), NOT the hour — a sustained blank-output condition
-- is one incident. The alert check thresholds (>2% rate, >3 blank, >=10 total) are
-- applied here so the findings table contains only actionable rows.
-- Currently verified as genuinely 0 rows. Wired so it works when it fires.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.blank_output_findings AS
SELECT
    capability, model_config,
    sum(blank_output_count) AS blank_count_window,
    sum(total_successful_calls) AS total_calls_window,
    round(sum(blank_output_count) * 1.0 / nullif(sum(total_successful_calls), 0), 4) AS blank_rate_window,
    max(hour) AS latest_hour,
    -- finding_signature: dedup key for obs_incidents' MERGE, keyed on the condition
    -- (capability+model_config) rather than any single row, so a sustained blank-output run is
    -- one incident whose detection_count climbs.
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(model_config, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.blank_output_incidents
WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL 6 HOURS)
GROUP BY capability, model_config
HAVING sum(blank_output_count) > 3
   AND sum(total_successful_calls) >= 10
   AND sum(blank_output_count) * 1.0 / nullif(sum(total_successful_calls), 0) > 0.02;

In [0]:
%sql
-- transport_violations -- flags calls using a transport/model_config/scheduler_run combination
-- not sanctioned for that capability. Feeds the runtime_violation detector, currently silenced
-- to WARN in ptof_obs_alert because runtime_allowlist doesn't yet reflect real SAA/ISH
-- production combinations with confidence (deprioritized for v1.1 reseeding).
-- Scoped to active capabilities via capability_registry join so deactivated DSA/probe
-- capabilities don't produce violation findings.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.transport_violations AS
WITH env_sanctioned AS (
  SELECT
      array_distinct(flatten(collect_list(allowed_transports)))     AS ok_transports,
      array_distinct(flatten(collect_list(allowed_model_configs)))  AS ok_model_configs,
      array_distinct(flatten(collect_list(allowed_scheduler_runs))) AS ok_scheduler_runs
  FROM mq_gmdf_dev.oil_obs.runtime_allowlist
  WHERE environment = :env
),
flagged AS (
  SELECT
      b.id, b.shift_date, b.shift_type, b.batch_nbr,
      b.capability, b.scheduler_run, b.transport, b.model_config, b.called_at,
      CASE
        WHEN a.capability IS NULL                                          THEN 'unknown_capability'
        WHEN NOT array_contains(a.allowed_transports,      b.transport)     THEN 'transport_not_allowed'
        WHEN NOT array_contains(a.allowed_model_configs,   b.model_config)  THEN 'model_config_not_allowed'
        WHEN NOT array_contains(a.allowed_scheduler_runs,  b.scheduler_run) THEN 'scheduler_run_not_allowed'
      END AS violation_type
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LEFT JOIN (SELECT * FROM mq_gmdf_dev.oil_obs.runtime_allowlist WHERE environment = :env) a
         ON a.capability = b.capability
  WHERE b.called_at >= current_timestamp() - INTERVAL 60 MINUTES
    AND (a.capability IS NULL
         OR NOT array_contains(a.allowed_transports,     b.transport)
         OR NOT array_contains(a.allowed_model_configs,  b.model_config)
         OR NOT array_contains(a.allowed_scheduler_runs, b.scheduler_run))
)
SELECT
    f.id, f.shift_date, f.shift_type, f.batch_nbr,
    f.capability, f.scheduler_run, f.transport, f.model_config, f.called_at,
    f.violation_type,
    sha2(concat_ws('|',
        coalesce(f.capability,     '<null>'),
        coalesce(f.transport,      '<null>'),
        coalesce(f.model_config,   '<null>'),
        coalesce(f.scheduler_run,  '<null>'),
        coalesce(f.violation_type, '<null>')
    ), 256) AS violation_signature,
    CASE
      WHEN size(coalesce(s.ok_transports, array())) = 0                            THEN 'digest'
      WHEN NOT array_contains(s.ok_transports,    coalesce(f.transport,    '<null>')) THEN 'immediate'
      WHEN NOT array_contains(s.ok_model_configs, coalesce(f.model_config, '<null>')) THEN 'immediate'
      ELSE 'digest'
    END AS violation_tier,
    current_timestamp() AS detected_at
FROM flagged f
CROSS JOIN env_sanctioned s;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- transport_violation_signatures — one row per DISTINCT configuration in the window. Exists
-- because obs_incidents MERGEs on (detector, source_row_id): a source with repeated keys throws
-- MULTIPLE_SOURCE_ROWS_MATCHED. The GROUP BY is exactly the sha2 input, so this is guaranteed
-- 1:1 with violation_signature. ptof_obs_weekly_runtime_digest reads this for the digest-tier
-- rollup; the immediate tier (runtime_violation) currently reads it too but is silenced to WARN.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.transport_violation_signatures AS
SELECT
    violation_signature,
    violation_tier,
    violation_type,
    capability, transport, model_config, scheduler_run,
    count(*)            AS calls_in_window,
    min(called_at)      AS first_called_at,
    max(called_at)      AS last_called_at,
    max(id)             AS sample_row_id,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.transport_violations
GROUP BY violation_signature, violation_tier, violation_type,
         capability, transport, model_config, scheduler_run;

In [ ]:
%sql
-- response_schema_drift -- detects a field that used to reliably appear in a capability's
-- response silently disappearing. This is what ptof_obs_alert's schema_field_missing detector
-- (CRITICAL in v1) reads (filtered to drift_type = 'field_missing').
-- Capability-level comparison only: per SME, model_config does not meaningfully affect response
-- shape for SAA/ISH agents. The two-tier pair-isolated/fallback structure was reverted as part
-- of the SAA/ISH rescope (6 extra CTEs removed, maintenance burden eliminated).
-- model_config kept as attribution via collect_set so findings say which configs had traffic.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_schema_drift AS
WITH baseline_keys AS (
  SELECT b.capability, k.key AS field_name, count(*) AS baseline_present
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LATERAL VIEW explode(from_json(cast(b.response_parsed AS STRING), 'map<string,string>')) k AS key, value
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  GROUP BY b.capability, k.key
),
baseline_rows AS (
  SELECT b.capability, count(*) AS n_rows
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  GROUP BY b.capability
),
current_keys AS (
  SELECT b.capability, k.key AS field_name, count(*) AS current_present,
         collect_set(b.model_config) AS model_configs_seen
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LATERAL VIEW explode(from_json(cast(b.response_parsed AS STRING), 'map<string,string>')) k AS key, value
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY b.capability, k.key
),
current_rows AS (
  SELECT b.capability, count(*) AS n_rows
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY b.capability
),
comparison AS (
  SELECT
      coalesce(bk.capability, ck.capability) AS capability,
      coalesce(bk.field_name, ck.field_name) AS field_name,
      bk.baseline_present,
      br.n_rows AS baseline_rows,
      round(bk.baseline_present * 1.0 / br.n_rows, 3) AS baseline_presence_rate,
      coalesce(ck.current_present, 0) AS current_present,
      ck.model_configs_seen,
      cr.n_rows AS current_rows,
      CASE WHEN bk.field_name IS NULL THEN 'field_added'
           ELSE 'field_missing' END AS drift_type,
      true AS schema_changed
  FROM baseline_keys bk
  FULL JOIN current_keys ck
    ON ck.capability = bk.capability AND ck.field_name = bk.field_name
  LEFT JOIN baseline_rows br ON br.capability = coalesce(bk.capability, ck.capability)
  LEFT JOIN current_rows  cr ON cr.capability = coalesce(bk.capability, ck.capability)
)
SELECT
    capability,
    field_name,
    baseline_present,
    baseline_rows,
    baseline_presence_rate,
    current_present,
    model_configs_seen,
    current_rows,
    drift_type,
    schema_changed,
    -- finding_signature: capability + field_name only (model_config removed from signature).
    -- Deliberate dedup-history reset: old per-config signatures won't match.
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(field_name, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM comparison
WHERE current_rows >= 10
  AND (drift_type = 'field_added'
       OR (current_present = 0 AND baseline_presence_rate >= 0.2));